In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
df = pd.read_csv('creditcard.csv')
df = df.drop(['Time'],axis=1)
df['Amount'] = np.log1p(df['Amount'])

In [3]:
data_left, test_data = train_test_split(df, test_size=0.2,stratify=df['Class'], random_state=43 )

In [4]:
fraud = data_left[data_left['Class']==1]
nfraud =  data_left[data_left['Class']==0]

In [5]:
nonf = nfraud.sample(n=24500, random_state=43)

In [6]:
train_data = pd.concat([fraud,nonf]).sample(frac=1,random_state=43)

In [7]:
train_data.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
177787,-1.960906,2.537838,-2.085075,-0.628376,-0.408078,-0.847051,-0.539551,1.725120,-0.537521,-0.659234,...,-0.193132,-0.744922,0.343946,0.668055,-0.190128,0.106881,0.041776,-0.003939,1.091923,0
232348,1.643182,-0.436400,-1.949938,1.336762,0.496877,-0.525566,0.793184,-0.301550,-0.065746,0.335727,...,0.192650,0.318814,-0.233729,-0.448407,0.413549,-0.502938,-0.054713,-0.042653,5.305789,0
177399,2.198154,-0.459232,-1.430987,-0.830930,-0.175329,-0.752800,-0.451662,-0.268160,-0.413898,0.165893,...,-0.360866,-0.984512,0.346954,-0.895632,-0.399945,-0.630858,0.017204,-0.020610,3.042139,0
220479,2.086891,0.060310,-1.504919,0.207094,0.413408,-0.765784,0.254550,-0.336375,0.431014,-0.031163,...,0.248347,0.905994,0.014564,0.667935,0.325378,-0.135197,-0.014485,-0.051327,0.559616,0
268337,0.019901,0.856890,0.191502,-0.794163,0.720514,-0.509913,0.936578,-0.055043,-0.393524,-0.271007,...,-0.216954,-0.435710,0.011854,-0.382321,-0.466399,0.143907,0.251302,0.084784,1.187843,0


In [8]:
test_data.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
169754,2.090345,0.059482,-1.871401,0.435197,0.682977,-0.747860,0.543390,-0.401785,0.219081,0.093875,...,-0.036045,0.095425,0.044826,0.549903,0.305484,0.358026,-0.084371,-0.065905,2.889816,0
19255,-1.497064,-0.308955,1.257550,-1.021315,0.038056,0.730570,-0.723600,1.048693,-0.909608,-0.254123,...,0.099546,0.487374,0.269355,-0.738866,-0.966102,-0.640927,-0.000878,0.109268,3.044522,0
142156,1.252514,0.100061,0.737343,0.522970,-0.526139,-0.427689,-0.285970,-0.084853,0.284316,-0.109726,...,-0.106982,-0.267573,0.016970,-0.084325,0.254340,0.241441,-0.002131,0.023911,2.385086,0
155839,-1.127011,1.490889,-1.044273,-0.368242,0.278635,-0.243068,1.071416,-0.202238,1.400308,-0.307763,...,0.248608,1.147557,0.086554,0.421521,-0.736203,-0.332041,-0.338976,0.257021,4.714473,0
108513,-1.417376,1.574589,0.273297,-0.989259,0.001513,-0.007290,-0.011552,0.970771,-0.487124,-0.662547,...,-0.152501,-0.434994,-0.133108,-0.795667,0.045087,0.354165,0.160632,0.094340,1.609438,0


In [9]:
category_to_rows = {}
train_data = train_data.reset_index(drop=True)
for row_index, category in enumerate(train_data["Class"]):

    key = ("Class", category)

    if key not in category_to_rows:
        category_to_rows[key] = []

    category_to_rows[key].append(row_index)

In [10]:
continuous_cols = [c for c in train_data.columns if c != 'Class']  # should be 30 cols after dropping Time
continuous_data = train_data[continuous_cols].to_numpy(dtype=np.float32)

# scale continuous columns to roughly [-1, 1] since Generator uses tanh on alpha_outputs
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
continuous_data = scaler.fit_transform(continuous_data)

# one-hot encode Class (2 categories) to match discrete_sizes=[2]
discrete_data = np.eye(2)[train_data['Class'].to_numpy()]

transformed_data = np.concatenate([continuous_data, discrete_data], axis=1).astype(np.float32)

In [11]:
print(category_to_rows[('Class',1)])

[30, 63, 193, 268, 279, 284, 319, 330, 483, 490, 520, 602, 761, 960, 1039, 1062, 1068, 1117, 1119, 1221, 1226, 1293, 1536, 1551, 1628, 1645, 1647, 1691, 1731, 1781, 1882, 2057, 2156, 2195, 2342, 2378, 2432, 2468, 2482, 2495, 2519, 2523, 2641, 2677, 2690, 2709, 2753, 2851, 2894, 2907, 2916, 2942, 3074, 3105, 3294, 3306, 3323, 3421, 3519, 3536, 3660, 3819, 3857, 3894, 3916, 4096, 4173, 4225, 4438, 4524, 4664, 4675, 4680, 4755, 5154, 5172, 5284, 5345, 5382, 5730, 5776, 5883, 5924, 5945, 6069, 6180, 6201, 6206, 6222, 6274, 6285, 6315, 6339, 6371, 6606, 6707, 6714, 6755, 6782, 6877, 6932, 7019, 7443, 7527, 7725, 7941, 8006, 8013, 8091, 8179, 8262, 8354, 8389, 8403, 8432, 8452, 8494, 8523, 8563, 8585, 8595, 8603, 8622, 8651, 8656, 8782, 8789, 8849, 8993, 8999, 9041, 9449, 9483, 9583, 9710, 9774, 9781, 9887, 9922, 9955, 9973, 10031, 10074, 10102, 10120, 10126, 10141, 10169, 10209, 10224, 10249, 10271, 10286, 10301, 10375, 10438, 10514, 10518, 10528, 10551, 10580, 10617, 10634, 10663, 10669, 1

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Generator(nn.Module):
    def __init__(
        self,
        embedding_dim,
        cond_dim,
        num_continuous,
        mode_sizes,
        discrete_sizes
    ):
        """
        embedding_dim : Dimension of latent vector z
        cond_dim      : Dimension of conditional vector
        num_continuous: Number of continuous columns
        mode_sizes    : List containing number of GMM modes for each
                        continuous column
                        Example: [5, 4, 3]

        discrete_sizes: List containing number of categories for each
                        categorical column
                        Example: [2, 4, 3]
        """

        super().__init__()

        self.embedding_dim = embedding_dim
        self.cond_dim = cond_dim

        input_dim = embedding_dim + cond_dim

        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)

        self.fc2 = nn.Linear(input_dim + 256, 256)
        self.bn2 = nn.BatchNorm1d(256)

        final_dim = input_dim + 256 + 256


        self.alpha_heads = nn.ModuleList([
            nn.Linear(final_dim, 1)
            for _ in range(num_continuous)
        ])

        self.beta_heads = nn.ModuleList([
            nn.Linear(final_dim, modes)
            for modes in mode_sizes
        ])

        self.discrete_heads = nn.ModuleList([
            nn.Linear(final_dim, categories)
            for categories in discrete_sizes
        ])

    def forward(self, z, cond):


        h0 = torch.cat([z, cond], dim=1)

        x = self.fc1(h0)
        x = self.bn1(x)
        x = F.relu(x)

        h1 = torch.cat([h0, x], dim=1)

        x = self.fc2(h1)
        x = self.bn2(x)
        x = F.relu(x)

        h2 = torch.cat([h1, x], dim=1)

        alpha_outputs = []

        for head in self.alpha_heads:

            alpha_outputs.append(
                torch.tanh(head(h2))
            )

        beta_outputs = []

        for head in self.beta_heads:

            beta_outputs.append(

                F.gumbel_softmax(
                    head(h2),
                    tau=0.2,
                    hard=False
                )

            )


        discrete_outputs = []

        for head in self.discrete_heads:

            discrete_outputs.append(

                F.gumbel_softmax(
                    head(h2),
                    tau=0.2,
                    hard=False
                )

            )
        return (
            torch.cat(alpha_outputs+beta_outputs+discrete_outputs, dim=1)
        )

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Critic(nn.Module):

    def __init__(
        self,
        data_dim,
        cond_dim,
        pac=1
    ):
        """
        data_dim : Dimension of transformed tabular data

        cond_dim : Dimension of conditional vector

        pac : PacGAN packing factor (default = 10)
        """

        super().__init__()

        self.pac = pac
        self.data_dim = data_dim
        self.cond_dim = cond_dim

        ########################################
        # Input dimension after packing
        ########################################

        input_dim = (data_dim + cond_dim) * pac

        ########################################
        # First Hidden Layer
        ########################################

        self.fc1 = nn.Linear(input_dim, 256)

        self.act1 = nn.LeakyReLU(0.2)

        self.drop1 = nn.Dropout(0.5)

        ########################################
        # Second Hidden Layer
        ########################################

        self.fc2 = nn.Linear(256, 256)

        self.act2 = nn.LeakyReLU(0.2)

        self.drop2 = nn.Dropout(0.5)

        ########################################
        # Output Layer
        ########################################

        self.fc3 = nn.Linear(256, 1)

    def forward(self, data, cond):

        ########################################
        # Concatenate condition
        ########################################

        x = torch.cat([data, cond], dim=1)

        ########################################
        # PacGAN Packing
        ########################################

        batch_size = x.size(0)

        assert batch_size % self.pac == 0, \
            "Batch size must be divisible by pac."

        x = x.view(
            batch_size // self.pac,
            self.pac * (self.data_dim + self.cond_dim)
        )

        ########################################
        # Layer 1
        ########################################

        x = self.fc1(x)
        x = self.act1(x)
        x = self.drop1(x)

        ########################################
        # Layer 2
        ########################################

        x = self.fc2(x)
        x = self.act2(x)
        x = self.drop2(x)

        ########################################
        # Output Score
        ########################################

        score = self.fc3(x)

        return score

In [14]:
class DataSampler:
    def __init__(self, dataframe, transformed_data, category_to_rows):
        self.df = dataframe
        self.data = transformed_data
        self.category_to_rows = category_to_rows

        counts = dataframe["Class"].value_counts().sort_index()
        log_freq = np.log(counts.values)
        self.pmf = log_freq / log_freq.sum()
        self.categories = counts.index.to_numpy()

    def sample(self, batch_size):
        chosen_categories = np.random.choice(
            self.categories, size=batch_size, p=self.pmf
        )
        idx = [
            self.category_to_rows[("Class", c)][
                np.random.randint(len(self.category_to_rows[("Class", c)]))
            ]
            for c in chosen_categories
        ]

        cond = np.zeros((batch_size, len(self.categories)), dtype=np.float32)
        cond[np.arange(batch_size), chosen_categories] = 1.0

        return (
            torch.tensor(self.data[idx], dtype=torch.float32),
            torch.tensor(cond, dtype=torch.float32),
        )

In [15]:
embedding_dim = 128
cond_dim = 2
data_dim = 32

In [16]:
import torch.optim as optim

generator = Generator(embedding_dim=128,
    cond_dim=2,
    num_continuous=len(continuous_cols),
    mode_sizes=[],
    discrete_sizes=[2])
critic = Critic(data_dim=transformed_data.shape[1],cond_dim=2)

generator_optimizer = optim.Adam(
    generator.parameters(),
    lr=2e-4,
    betas=(0.5, 0.9)
)

critic_optimizer = optim.Adam(
    critic.parameters(),
    lr=2e-4,
    betas=(0.5, 0.9)
)

In [17]:
import torch
from torch.autograd import grad


def gradient_penalty(critic,
                     real_data,
                     fake_data,
                     cond):

    batch_size = real_data.size(0)

    alpha = torch.rand(batch_size, 1)

    alpha = alpha.expand_as(real_data)

    interpolated = alpha * real_data + (1 - alpha) * fake_data

    interpolated.requires_grad_(True)

    score = critic(interpolated, cond)

    gradients = grad(
        outputs=score,
        inputs=interpolated,
        grad_outputs=torch.ones_like(score),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients = gradients.view(batch_size, -1)

    gp = ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    return 10 * gp

In [18]:
sampler = DataSampler(
    train_data,
    transformed_data,
    category_to_rows
)
num_epochs=100
embedding_dim=128
for epoch in range(num_epochs):

    ##################################################
    # Train Critic
    ##################################################

    for _ in range(5):

        real_batch, cond_batch = sampler.sample(256)

        noise = torch.randn(
            256,
            embedding_dim
        )

        fake_batch = generator(
            noise,
            cond_batch
        )

        real_score = critic(
            real_batch,
            cond_batch
        )

        fake_score = critic(
            fake_batch.detach(),
            cond_batch
        )

        gp = gradient_penalty(
            critic,
            real_batch,
            fake_batch,
            cond_batch
        )

        critic_loss = (
            fake_score.mean()
            -
            real_score.mean()
            +
            gp
        )

        critic_optimizer.zero_grad()

        critic_loss.backward()

        critic_optimizer.step()

    ##################################################
    # Train Generator
    ##################################################

    real_batch, cond_batch = sampler.sample(256)

    noise = torch.randn(
        256,
        embedding_dim
    )

    fake_batch = generator(
        noise,
        cond_batch
    )

    fake_score = critic(
        fake_batch,
        cond_batch
    )

    generator_loss = -fake_score.mean()

    generator_optimizer.zero_grad()

    generator_loss.backward()

    generator_optimizer.step()

    print(
        f"Epoch {epoch}",
        f"Critic {critic_loss.item():.4f}",
        f"Generator {generator_loss.item():.4f}"
    )

Epoch 0 Critic 6.0337 Generator 0.0169
Epoch 1 Critic 5.0443 Generator 0.0080
Epoch 2 Critic 3.7763 Generator 0.0027
Epoch 3 Critic 2.6397 Generator -0.0351
Epoch 4 Critic 1.6756 Generator -0.1042
Epoch 5 Critic 0.8993 Generator -0.2239
Epoch 6 Critic 0.5771 Generator -0.3189
Epoch 7 Critic 0.4198 Generator -0.4570
Epoch 8 Critic 0.3318 Generator -0.5349
Epoch 9 Critic 0.2668 Generator -0.6108
Epoch 10 Critic 0.2055 Generator -0.6291
Epoch 11 Critic 0.1211 Generator -0.5558
Epoch 12 Critic -0.0834 Generator -0.5066
Epoch 13 Critic -0.2344 Generator -0.3892
Epoch 14 Critic -0.3147 Generator -0.3021
Epoch 15 Critic -0.3769 Generator -0.1491
Epoch 16 Critic -0.5803 Generator 0.0256
Epoch 17 Critic -0.5978 Generator 0.1082
Epoch 18 Critic -0.7393 Generator 0.2195
Epoch 19 Critic -0.7546 Generator 0.3083
Epoch 20 Critic -0.8466 Generator 0.3698
Epoch 21 Critic -0.8772 Generator 0.4518
Epoch 22 Critic -0.7938 Generator 0.5298
Epoch 23 Critic -0.7880 Generator 0.5309
Epoch 24 Critic -0.7396 G

In [21]:
generator.eval()  # turn off BatchNorm/Dropout randomness

num_synthetic = 20000  # however many extra fraud rows you want

with torch.no_grad():
    noise = torch.randn(num_synthetic, embedding_dim)
    cond = torch.zeros(num_synthetic, cond_dim)
    cond[:, 1] = 1.0   # one-hot for Class=1 (fraud) — matches your cond encoding in DataSampler

    synthetic_batch = generator(noise, cond)
num_continuous=len(continuous_cols)

In [22]:
synthetic_np = synthetic_batch.numpy()

continuous_part = synthetic_np[:, :num_continuous]      # first 29 cols
discrete_part = synthetic_np[:, num_continuous:]         # last 2 cols (softmax over Class)

# invert the MinMaxScaler
continuous_original = scaler.inverse_transform(continuous_part)

# discrete part -> hard class label (argmax of softmax)
synthetic_class = discrete_part.argmax(axis=1)  # should be all 1s since we conditioned on fraud

synthetic_df = pd.DataFrame(continuous_original, columns=continuous_cols)
synthetic_df['Class'] = synthetic_class

In [23]:
augmented_train = pd.concat(
    [train_data[continuous_cols + ['Class']], synthetic_df],
    ignore_index=True
).sample(frac=1, random_state=43)  # shuffle

In [25]:
synthetic_df.head()
synthetic_df['Class'].value_counts()

Class
0    11098
1     8902
Name: count, dtype: int64

In [26]:
synthetic_df.tail()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
19995,-0.899812,7.238269,-2.582833,2.899647,-7.216470,5.000343,-11.493918,1.820590,-2.341367,-14.474329,...,7.944237,-0.033006,-4.039002,-0.350729,-0.625836,0.828237,0.846679,1.769244,3.277314,0
19996,1.296757,-2.647107,-2.924233,4.492483,7.828479,2.735658,-5.694346,-5.680162,-0.641482,-2.301200,...,5.532394,1.879969,-10.434633,-0.886229,0.163449,1.644657,-1.615468,2.793245,3.400641,1
19997,0.190872,-2.826656,-0.026275,1.502640,2.890816,-2.458103,4.625217,-3.243571,1.750677,2.539450,...,1.395996,0.475724,6.561207,-0.680032,0.271625,-0.349744,-1.754790,0.426169,3.927054,0
19998,-6.352119,8.596915,-4.888396,6.808392,-3.079046,5.622208,-20.906219,4.878383,-1.825655,-6.266958,...,-2.296556,-1.813480,3.643325,-1.365520,-0.502074,0.415022,-3.384949,5.750087,3.504235,0
19999,0.559498,3.352482,-6.371337,5.623297,-0.034819,5.356679,-2.396955,2.094033,-5.731979,1.557652,...,11.022591,-0.031979,-7.832001,-0.362158,1.817739,0.554510,2.481643,-1.322098,2.668540,1
